# OpenAI-style messages and native Apertus: what conversion changes

Compare the actual representations, inspect conversion diagnostics, and see how a lossy round trip can change a training trace or a replayed inference prompt.

Here **OpenAI-style** means this adapter's supported chat-message subset. `reasoning_content` is a compatibility extension, not a universal OpenAI API field. Native JSON stores Apertus conversations; model inputs are the token sequence produced by the Apertus encoder. An OpenAI-shaped JSON object alone does not specify any model's token template.

Run `make install` first. The format examples need no tokenizer or model. Set `APERTUS_ENCODING` to a local Apertus 2 instruct artifact for the final token/weight examples, then run all cells or `make examples`. No model is trained or queried here; the examples establish representation changes, not measured model-quality effects.

[Implemented mapping contract](../docs/design.md#openai-mapping) · [Encoding and weights](../docs/design.md#rendering-and-encoding)

In [ ]:
import json
import os
from collections.abc import Callable, Mapping, Sequence
from dataclasses import asdict

from apertus_common import (
    AdapterError,
    Call,
    Claim,
    Conversation,
    Document,
    Host,
    MappingDiagnostic,
    MappingPolicy,
    MappingResult,
    OpenAIMessage,
    OpenAIRequest,
    Reply,
    Result,
    SystemPrompt,
    Think,
    UnknownOutput,
    User,
    Wait,
    checkers_for,
    from_openai,
    load_encoding,
    render_str,
    to_openai,
    to_openai_output,
)


def show(label: str, value: object) -> None:
    print(label)
    print(json.dumps(value, ensure_ascii=False, indent=2))


def show_diagnostics(issues: Sequence[MappingDiagnostic]) -> None:
    show("Diagnostics", [asdict(issue) for issue in issues])


def rejected(action: Callable[[], object]) -> None:
    try:
        action()
    except AdapterError as error:
        print(f"Rejected: {error}")
    else:
        raise AssertionError("Expected this conversion to be rejected")


def replay(projected: MappingResult[OpenAIRequest]) -> Conversation:
    request = projected.value
    return from_openai(
        request.get("messages", []),
        tools=request.get("tools"),
        reasoning_effort=request.get("reasoning_effort"),
        context=projected.context,
    ).value

## 1. Same tool exchange, different structure

| Concern | OpenAI-style chat request | Native Apertus |
| --- | --- | --- |
| Messages | `role` plus content/assistant fields | Ordered typed inputs and outputs in `items` |
| System settings | Leading message and request options | Separate structured `system` object |
| Reasoning and answer | Fields on an assistant record | Separate `Think` and `Reply` messages |
| Tools | Request-level function definitions | Declarations in `system.tools` |
| Tool association | String `tool_call_id` | Conversation-wide integer counter and tool name |
| Giving control back | Represented through chat/API turn conventions | Explicit `Wait` ending an output burst |
| Extra native content | No counterpart in this adapter for host/document/claim | Dedicated types with their own meaning |

The mapper below creates the default native thinking setting and adds waits after completed assistant runs. It preserves the raw argument string, including its whitespace. Mapping and checking are separate operations.

In [ ]:
tools: list[Mapping[str, object]] = [
    {
        "type": "function",
        "function": {
            "name": "weather",
            "description": "Weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    }
]
chat: list[OpenAIMessage] = [
    {"role": "system", "content": "Be concise."},
    {"role": "user", "content": "What is the weather in Zurich?"},
    {
        "role": "assistant",
        "content": None,
        "reasoning_content": "Check the weather tool.",
        "tool_calls": [
            {
                "id": "call_weather",
                "type": "function",
                "function": {"name": "weather", "arguments": '{"city": "Zurich"}'},
            }
        ],
    },
    {"role": "tool", "tool_call_id": "call_weather", "content": '{"tempC":20}'},
    {"role": "assistant", "content": "It is 20 degrees Celsius."},
]
source = from_openai(chat, tools=tools, reasoning_effort="medium")
checked = source.value.check()
assert checked.ok and checked.complete and not source.diagnostics
assert [item.type for item in source.value.items] == [
    "user",
    "think",
    "call",
    "wait",
    "result",
    "reply",
    "wait",
]
show(
    "OpenAI-style request",
    {"messages": chat, "tools": tools, "reasoning_effort": "medium"},
)
show("Native storage", json.loads(source.value.to_json()))
print("Native wire view (diagnostic text, not the JSON sent to a model):")
print(render_str(source.value))

## 2. Preserved content versus exact representation

For the supported exchange above, keeping the mapping context preserves the original tool IDs as well as the text and arguments. Native JSON carries counters, not those external IDs: persist the context separately if you need exact external IDs across requests or history compaction.

This successful example is not a guarantee that arbitrary JSON round trips unchanged. Missing fields, defaults, content-part boundaries and assistant boundaries need separate treatment.

In [ ]:
round_trip = to_openai(source.value, context=source.context)
assert round_trip.value.get("messages") == chat
assert not round_trip.diagnostics
assert replay(round_trip) == source.value
show("Retained external tool IDs", source.context.model_dump(mode="json"))

regenerated = to_openai(source.value)
assert regenerated.context.associations[0].id == "apertus_0"
assert source.context.associations[0].id == "call_weather"
print(
    "Without context, a deterministic replacement ID is allocated:",
    regenerated.context.associations[0].id,
)

### Normalization: text survives, container shape changes

Text-part arrays become concatenated strings, and an assistant with no reasoning, content or calls becomes an empty reply followed by a wait. The library reports these as normalizations (`lossy=False`), without a policy flag. That label does not promise preservation of source JSON shape or text-part boundaries.

Likewise, missing/null tool descriptions and parameter schemas receive defaults; `strict: false` is accepted, while `strict: true` is rejected.

In [ ]:
parts_input: list[Mapping[str, object]] = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Hello "},
            {"type": "text", "text": "world"},
        ],
    }
]
normalized = from_openai(parts_input)
assert normalized.value.items == [User(payload="Hello world")]
assert [issue.code for issue in normalized.diagnostics] == ["text-parts-joined"]
assert not any(issue.lossy for issue in normalized.diagnostics)
show("Before", parts_input)
show("After", to_openai(normalized.value).value.get("messages"))
show_diagnostics(normalized.diagnostics)

empty = from_openai([{"role": "assistant", "content": None}])
assert empty.value.items == [Reply(payload=""), Wait()]
assert [issue.code for issue in empty.diagnostics] == ["empty-assistant"]
show("Empty assistant after round trip", to_openai(empty.value).value.get("messages"))

## 3. Lossy import: system/developer distinctions become one behavior string

One leading system message maps to native `system.behavior`. Multiple leading system/developer messages require `merge_system_messages=True`. The adapter joins them with blank lines and reports the lost role/priority distinction. It cannot reconstruct that distinction on export.

For training, the resulting native system prompt contains one instruction block. Any task depending on separate instruction priorities needs an explicit authoring decision; concatenation alone supplies no separate native roles. Later system/developer messages are rejected even with this policy.

In [ ]:
instructions: list[OpenAIMessage] = [
    {"role": "system", "content": "Follow the safety policy."},
    {"role": "developer", "content": "Use concise sentences."},
    {"role": "user", "content": "Hello"},
]
rejected(lambda: from_openai(instructions))
merged_instructions = from_openai(
    instructions, policy=MappingPolicy(merge_system_messages=True)
)
assert merged_instructions.value.system is not None
assert merged_instructions.value.system.behavior == (
    "Follow the safety policy.\n\nUse concise sentences."
)
assert any(issue.lossy for issue in merged_instructions.diagnostics)
show("Before", instructions)
show("After round trip", to_openai(merged_instructions.value).value.get("messages"))
show_diagnostics(merged_instructions.diagnostics)

## 4. Lossy response projection: output order and boundaries

A native burst can contain `think → reply → think → reply → wait`. History conversion can preserve this example by splitting it into two assistant records; it does not need a lossy policy here.

A **single assistant response**, however, has one reasoning field and one content field. `to_openai_output` rejects that burst by default. With `merge_assistant_messages=True`, it groups reasoning and reply segments into their respective fields. Importing that assembled response puts all reasoning before all replies, changing the native trace.

Streaming deltas can arrive in their original order, but storing only the assembled response fields loses the cross-field ordering. Replaying that flattened message during inference or training presents a different sequence to the model.

In [ ]:
ordered_output = Conversation(
    items=[
        Think(payload="Check one.\n"),
        Reply(payload="Answer one.\n"),
        Think(payload="Check two.\n"),
        Reply(payload="Answer two.\n"),
        Wait(),
    ]
)
ordered_history = Conversation(
    system=SystemPrompt.build(),
    items=[User(payload="Compare two cases."), *ordered_output.items],
)
assert ordered_history.check().ok
history_projection = to_openai(ordered_history)
assert not history_projection.diagnostics
assert replay(history_projection) == ordered_history
show(
    "History projection preserves this order with two assistant records",
    history_projection.value.get("messages"),
)

rejected(lambda: to_openai_output(ordered_output))
flat_response = to_openai_output(
    ordered_output, policy=MappingPolicy(merge_assistant_messages=True)
)
assert flat_response.value.get("reasoning_content") == "Check one.\nCheck two.\n"
assert flat_response.value.get("content") == "Answer one.\nAnswer two.\n"
flattened_output = from_openai([flat_response.value]).value
assert [item.type for item in flattened_output.items] == ["think", "reply", "wait"]
assert any(issue.lossy for issue in flat_response.diagnostics)
flattened_history = Conversation(
    system=ordered_history.system,
    items=[User(payload="Compare two cases."), *flattened_output.items],
)
print("Before:", render_str(ordered_output))
show("Assembled response", flat_response.value)
print("After replay:", render_str(flattened_output))
show_diagnostics(flat_response.diagnostics)

## 5. Lossy history projection: input during an output burst

A ready tool result may be supplied by the harness at a message boundary before the model emits `Wait`. This is valid native history. The training checker warns because ordinary turn-based datasets usually intend a wait, so keep this pattern only when the recipe and runtime intend it.

Chat history conversion requires the merging policy and inserts an assistant turn boundary before the result. Importing the projected history adds a wait after the call. The payloads survive, but the control-flow example changes from mid-burst delivery to a wait/resume exchange.

The model generates output envelopes. The harness supplies input envelopes and schedules delivery; a training trace containing `Result` does not authorize the model to generate that input itself.

In [ ]:
early_result = Conversation(
    system=source.value.system,
    items=[
        User(payload="What is the weather in Zurich?"),
        Call.build(name="weather", counter=0, arguments={"city": "Zurich"}),
        Result(name="weather", counter=0, payload='{"tempC":20}'),
        Reply(payload="It is 20 degrees Celsius."),
        Wait(),
    ],
)
early_report = early_result.check(checkers=checkers_for("full", training=True))
assert early_report.ok and early_report.complete
assert [warning.rule for warning in early_report.warnings] == [
    "training/input-mid-burst"
]
rejected(lambda: to_openai(early_result))
turn_based = to_openai(
    early_result, policy=MappingPolicy(merge_assistant_messages=True)
)
after_early_replay = replay(turn_based)
assert [item.type for item in after_early_replay.items] == [
    "user",
    "call",
    "wait",
    "result",
    "reply",
    "wait",
]
assert any(
    issue.code == "burst-boundary-inserted" and issue.lossy
    for issue in turn_based.diagnostics
)
print("Before:", render_str(early_result))
print("After:", render_str(after_early_replay))
show_diagnostics(turn_based.diagnostics)

## 6. Lossy export: claims and training weights

A claim can carry a separate verifier/grader target. It has no counterpart in this adapter's chat fields. Native loss weights are training annotations, also without a chat counterpart. Omission requires `omit_claims=True` and `omit_loss_weights=True` respectively; one flag does not authorize the other loss.

After the round trip below, the claim is absent and the reasoning is unannotated. Native default weighting now supervises that reasoning with weight 1 instead of its original 0. A zero weight suppresses direct loss on those tokens; the tokens still appear in the causal context for later output. Dropping a weight changes the training objective even when the retained text is identical.

In [ ]:
weighted = Conversation(
    system=SystemPrompt.build(),
    items=[
        User(payload="What is 6 times 7?"),
        Think(payload="Multiply the numbers.", loss_weight=0),
        Claim(payload="42", loss_weight=0.5),
        Reply(payload="42", loss_weight=1),
        Wait(loss_weight=1),
    ],
)
weight_report = weighted.check(
    checkers=checkers_for("full", allow_loss_weights=True, training=True)
)
assert weight_report.ok and weight_report.complete
rejected(lambda: to_openai(weighted))
rejected(lambda: to_openai(weighted, policy=MappingPolicy(omit_loss_weights=True)))
chat_only = to_openai(
    weighted, policy=MappingPolicy(omit_claims=True, omit_loss_weights=True)
)
weights_replayed = replay(chat_only)
assert not any(isinstance(item, Claim) for item in weights_replayed.items)
assert all(item.loss_weight is None for item in weights_replayed.items)
assert {issue.code for issue in chat_only.diagnostics} >= {
    "claim-omitted",
    "loss-weight-omitted",
}
show("Before: native storage retains annotations", json.loads(weighted.to_json()))
show("Chat projection", chat_only.value)
show("After replay", json.loads(weights_replayed.to_json()))
show_diagnostics(chat_only.diagnostics)

## 7. Some native features are rejected, even with every loss policy enabled

Host instructions, document provenance, unknown outputs and richer native system settings cannot be flattened by this adapter. There is no generic `allow_loss` switch. Text placed in an ordinary user message is not equivalent to a native host or document envelope with its own trust/provenance meaning.

Non-text chat parts and `strict: true` tool schemas are also rejected. Supporting those requires a separate implementation decision, not more permissive flags.

In [ ]:
all_losses = MappingPolicy(
    merge_system_messages=True,
    merge_assistant_messages=True,
    omit_claims=True,
    omit_loss_weights=True,
)
unsupported: dict[str, Conversation] = {
    "host": Conversation(
        system=SystemPrompt.build(), items=[Host(payload="A runtime instruction.")]
    ),
    "document": Conversation(
        system=SystemPrompt.build(),
        items=[Document(from_="retriever", id="doc1", payload="Evidence.")],
    ),
    "unknown output": Conversation(
        system=SystemPrompt.build(),
        items=[UnknownOutput(output_type="memo", payload="Extra.")],
    ),
    "richer system": Conversation(system=SystemPrompt.build(identity="Aria")),
}
for label, native_case in unsupported.items():
    try:
        to_openai(native_case, policy=all_losses)
    except AdapterError as error:
        print(f"{label}: {error}")
    else:
        raise AssertionError(f"Unexpectedly mapped {label}")

non_text: list[Mapping[str, object]] = [
    {
        "role": "user",
        "content": [{"type": "image_url", "image_url": {"url": "example.png"}}],
    }
]
rejected(lambda: from_openai(non_text))
strict_tools: list[Mapping[str, object]] = [
    {"type": "function", "function": {"name": "weather", "strict": True}}
]
rejected(lambda: from_openai([], tools=strict_tools))

## 8. What the model sees during training and inference

For Apertus training, convert source data to native records, explicitly check it, then encode it with the Apertus tokenizer. The model receives framed tokens, not the complete native storage object or OpenAI dictionaries printed above. The native system payload is compact JSON inside its input envelope; outer storage fields such as `schema_version` and `items` are not encoded as model text. Control-token spellings in these rendered views are diagnostic; ordinary payload text cannot inject those control IDs through the encoder.

```text
Full causal training trace:
BOS → system input → user input → think → call → wait → tool result input → reply → wait

Inference after the tool result:
prefix: BOS → system input → user input → think → call → wait → tool result input
generate: reply → wait
```

The [profile](https://github.com/swiss-ai/apertus-2-format-spec-proposal/blob/4f6f170731cfe79c226f3c777204be5e4cb69203/profile/apertus_2/apertus_2.md#context-and-memory-policy) trains on complete multi-turn traces with earlier thinking retained. The per-burst example below demonstrates continuation-only supervision with the full prefix preserved. **The encoder defaults to weight 1 for every output and wait, including earlier ones in the prefix.** The caller explicitly masks that prefix for this objective.

Input/system payloads default to weight 0; their framing always has weight 0. Explicit annotations can change payload weights. BOS is always weight 0. Output weights cover their entire frames. Weights are unshifted: the trainer must align them with its shifted targets and handle packing/padding. Setting a token's weight to 0 does not remove it from attention.

The importer treats assistant runs as completed and inserts waits. For interrupted outputs or deliberate mid-burst delivery, preserve native boundaries rather than treating every chat record as a completed turn.

In [ ]:
prompt_after_result = Conversation(
    system=source.value.system, items=source.value.items[:5]
)
continuation = Conversation(items=source.value.items[5:])
assert prompt_after_result.check().ok
assert continuation.items == [Reply(payload="It is 20 degrees Celsius."), Wait()]
print("Inference prefix / training context:")
print(render_str(prompt_after_result))
print("Next generated burst / supervised continuation:")
print(render_str(continuation))

artifact = os.environ.get("APERTUS_ENCODING")
encoding = load_encoding(artifact) if artifact else None
if encoding is not None:
    prefix_ids = encoding.encode_conversation(prompt_after_result)
    target = encoding.encode_items(continuation.items, return_loss_weights=True)
    full = encoding.encode_conversation(source.value, return_loss_weights=True)
    response_only_ids = prefix_ids + target.token_ids
    response_only_weights = [0.0] * len(prefix_ids) + target.loss_weights
    assert response_only_ids == full.token_ids
    assert len(response_only_weights) == len(response_only_ids)
    assert sum(response_only_weights) < sum(full.loss_weights)
    show(
        "Same tokens, different training objective",
        {
            "tokens": len(full.token_ids),
            "full_trace_weight_sum": sum(full.loss_weights),
            "continuation_only_weight_sum": sum(response_only_weights),
        },
    )
    assert encoding.generation_constraints.eos_token_id == encoding.controls.wait
else:
    print("Set APERTUS_ENCODING to verify the token/weight examples.")

### Verify that the demonstrated losses reach tokens and weights

Flattening ordered segments changes the token sequence. Dropping the reasoning weight leaves that reasoning's tokens unchanged but changes their supervision. These are separate effects.

In [ ]:
if encoding is not None:
    ordered_ids, flattened_ids = encoding.encode_conversations(
        [ordered_history, flattened_history]
    )
    assert ordered_ids != flattened_ids
    show(
        "Ordering changes model input",
        {
            "original_tokens": len(ordered_ids),
            "flattened_tokens": len(flattened_ids),
            "identical_ids": ordered_ids == flattened_ids,
        },
    )
    original_think = next(item for item in weighted.items if isinstance(item, Think))
    replayed_think = next(
        item for item in weights_replayed.items if isinstance(item, Think)
    )
    before_weights = encoding.encode_items([original_think], return_loss_weights=True)
    after_weights = encoding.encode_items([replayed_think], return_loss_weights=True)
    assert before_weights.token_ids == after_weights.token_ids
    assert set(before_weights.loss_weights) == {0.0}
    assert set(after_weights.loss_weights) == {1.0}
    show(
        "Identical reasoning tokens, changed loss",
        {
            "tokens": len(before_weights.token_ids),
            "weight_before": sorted(set(before_weights.loss_weights)),
            "weight_after": sorted(set(after_weights.loss_weights)),
        },
    )

## 9. Consequences to check in a training or serving recipe

| Conversion choice | What changes in the stored/replayed example | Possible consequence |
| --- | --- | --- |
| Merge system/developer messages | One behavior block replaces distinct roles | Role-priority supervision is no longer explicit |
| Flatten output segments into one response | Reasoning/reply order and boundaries | Different causal context on replay; less evidence for the original sequencing |
| Project mid-burst tool results to turn-based chat | A wait/resume boundary appears | Different examples of stopping and when results arrive |
| Omit claims | Separate verifier target disappears | A grader or objective depending on it needs another source |
| Omit weights | Default supervision replaces explicit annotations | Masked reasoning or downweighted answers may receive full loss |
| Lose tool-ID context | External IDs are regenerated | Application replay must use the new IDs consistently |

These are representation and objective differences, not evidence of a particular accuracy change. Evaluate converted training data against the actual inference prompt construction and scheduling behavior.

For native training/serving, retain native JSON and any needed mapping context, check records, then encode directly. For a model using another chat template, use that model's own verified tokenizer/template; this notebook does not define its wire format. At inference, apply the Apertus generation constraints, parse generated output, and let the harness execute tools and deliver inputs. A conformant but lossy conversion can still be unsuitable for a particular training objective.